In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import pandas as pd


data=pd.read_csv("../data/customer-churn-clustered.csv")
print(data.head())

print(data["Churn"].value_counts())
print(data["Cluster"].value_counts())



   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... TechSupport  \
0  No phone service             DSL             No  ...          No   
1                No             DSL            Yes  ...          No   
2                No             DSL            Yes  ...          No   
3  No phone service             DSL            Yes  ...         Yes   
4                No     Fiber optic             No  ...          No   

  StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No             

In [29]:

y=data["Churn"].map({
    "No":0,
    "Yes":1
})
print(y.value_counts())
X=data.drop(columns=["customerID","Churn"])
# print(x.columns)

# cluster -> str
X["Cluster"]=X["Cluster"].astype(str)

# split
# 
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )#la meme proportion de churn dans train et dans test (data desequilibre)

# separation du column

numeric_features=X_train.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

categorical_features=X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# print(numeric_features)
# print(categorical_features)


numeric_transformer = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

Churn
0    5174
1    1869
Name: count, dtype: int64


C:\Users\youco\AppData\Local\Temp\ipykernel_16916\1339943929.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features=X_train.select_dtypes(


In [30]:
# grid search

pipeline=ImbPipeline([
    ("preprocessing",preprocessor),
    ("smote",SMOTE(random_state=42)),
    ("model",LogisticRegression(max_iter=2000))
])

param_grid={
    "model__C":[0.01,0.1,1,10,100]
}

cv_strategy=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search=GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train,y_train)

print("le meilleur param est : ",grid_search.best_params_)
print("meilleur f1-score en validation croise",round(grid_search.best_score_,4))

# evaluer

best_logistic_with_cluster=grid_search.best_estimator_

y_pred=best_logistic_with_cluster.predict(X_test)
y_proba=best_logistic_with_cluster.predict_proba(X_test)[:,1]

result={
    "Model":"Logistic regression",
    "Version":"Apres opti avec cluster",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba)
}

print(result)
print(confusion_matrix(y_test,y_pred))
# print(y_pred)


# 2  meilleur model avec cluster (logistic regression + svm)
# 2 meilleur model sans cluster (logistic regression + xgb)


Fitting 5 folds for each of 5 candidates, totalling 25 fits
le meilleur param est :  {'model__C': 1}
meilleur f1-score en validation croise 0.6316
{'Model': 'Logistic regression', 'Version': 'Apres opti avec cluster', 'Accuracy': 0.7381121362668559, 'Precision': 0.5042444821731749, 'Recall': 0.7941176470588235, 'F1-score': 0.616822429906542, 'ROC-AUC': 0.8403032886408844}
[[743 292]
 [ 77 297]]


In [32]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

svm_pipeline = ImbPipeline([
    ("preprocessing", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", SVC(probability=True,random_state=42))
])

svm_param_grid = {
    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale"]
}

svm_grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2
)

svm_grid_search.fit(X_train, y_train)

print("Meilleurs parametres :", svm_grid_search.best_params_)
print("Meilleur F1 en validation croise :",round(svm_grid_search.best_score_, 4))

Fitting 3 folds for each of 6 candidates, totalling 18 fits


c:\CustomerRisk\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Meilleurs parametres : {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Meilleur F1 en validation croise : 0.6266


In [33]:
# optimisation sans cluster
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_train_no_cluster = X_train.drop(columns=["Cluster"])
X_test_no_cluster = X_test.drop(columns=["Cluster"])

numeric_features_no_cluster = (
    X_train_no_cluster
    .select_dtypes(include=["int64", "float64"])
    .columns.tolist()
)

categorical_features_no_cluster = (
    X_train_no_cluster
    .select_dtypes(include=["object", "category"])
    .columns.tolist()
)

numeric_transformer_no_cluster = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_no_cluster = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_no_cluster = ColumnTransformer([
    ("num", numeric_transformer_no_cluster, numeric_features_no_cluster),
    ("cat", categorical_transformer_no_cluster, categorical_features_no_cluster)
])

C:\Users\youco\AppData\Local\Temp\ipykernel_16916\758406814.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category"])


In [34]:
# LR
pipeline_lr_no_cluster = ImbPipeline([
    ("preprocessing", preprocessor_no_cluster),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=2000))
])

param_grid_lr_no_cluster = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid_lr_no_cluster = GridSearchCV(
    estimator=pipeline_lr_no_cluster,
    param_grid=param_grid_lr_no_cluster,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

grid_lr_no_cluster.fit(X_train_no_cluster, y_train)

print("Meilleurs parametres :", grid_lr_no_cluster.best_params_)
print("Meilleur F1 en validation croisee :",round(grid_lr_no_cluster.best_score_, 4))


Fitting 5 folds for each of 5 candidates, totalling 25 fits
Meilleurs parametres : {'model__C': 0.1}
Meilleur F1 en validation croisee : 0.633


In [35]:
# xgb
from xgboost import XGBClassifier

pipeline_xgb_no_cluster = ImbPipeline([
    ("preprocessing", preprocessor_no_cluster),
    ("smote", SMOTE(random_state=42)),
    ("model", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

param_grid_xgb = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.05, 0.1]
}

grid_xgb_no_cluster = GridSearchCV(
    estimator=pipeline_xgb_no_cluster,
    param_grid=param_grid_xgb,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

grid_xgb_no_cluster.fit(X_train_no_cluster, y_train)

print("Meilleurs parametres :", grid_xgb_no_cluster.best_params_)
print("Meilleur F1 en validation croisee :",round(grid_xgb_no_cluster.best_score_, 4))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Meilleurs parametres : {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}
Meilleur F1 en validation croisee : 0.6382


In [36]:
# evaluation
def evaluer_modele(modele, X_test, y_test, nom, version):
    predictions = modele.predict(X_test)
    probabilites = modele.predict_proba(X_test)[:, 1]

    return {
        "Modele": nom,
        "Version": version,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilites)
    }

resultats_optimises = [
    evaluer_modele(grid_search.best_estimator_,X_test, y_test,"Logistic Regression","Avec cluster"),

    evaluer_modele(svm_grid_search.best_estimator_,X_test, y_test,"SVM","Avec cluster"),

    evaluer_modele(grid_lr_no_cluster.best_estimator_,X_test_no_cluster, y_test,"Logistic Regression","Sans cluster"),

    evaluer_modele(grid_xgb_no_cluster.best_estimator_,X_test_no_cluster, y_test,"XGBoost","Sans cluster")
]


In [37]:
# model optimise
results_optimized = pd.DataFrame(resultats_optimises)

display(results_optimized.round(4))

results_optimized.to_csv(
    "../results/optimized_models_comparison.csv",
    index=False
)


,Modele,Version,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,Avec cluster,0.7381,0.5042,0.7941,0.6168,0.8403
1,SVM,Avec cluster,0.7445,0.5122,0.7861,0.6203,0.8289
2,Logistic Regression,Sans cluster,0.7438,0.5113,0.7861,0.6196,0.8397
3,XGBoost,Sans cluster,0.7708,0.5511,0.7353,0.6300,0.8428


In [38]:
import joblib

best_svm_with_cluster=svm_grid_search.best_estimator_
best_lr_no_cluster=grid_lr_no_cluster.best_estimator_
best_xgb_no_cluster=grid_xgb_no_cluster
joblib.dump(
    best_logistic_with_cluster,
    "../models/logistic_with_cluster_optimized.pkl"
)

joblib.dump(
    best_svm_with_cluster,
    "../models/svm_with_cluster_optimized.pkl"
)

joblib.dump(
    best_lr_no_cluster,
    "../models/logistic_without_cluster_optimized.pkl"
)

joblib.dump(
    best_xgb_no_cluster,
    "../models/xgboost_without_cluster_optimized.pkl"
)

['../models/xgboost_without_cluster_optimized.pkl']

In [ ]:
# charger les résultats initiaux
sans_cluster = pd.read_csv("../results/classification_without_cluster.csv")
avec_cluster = pd.read_csv("../results/classification_with_cluster.csv")

baseline_sans=sans_cluster[sans_cluster["Modele"].isin(["Logistic Regression","SVM"])].copy()
baseline_sans["Version"]="Avant -Sans cluster"
# print(baseline_sans)

baseline_avec = avec_cluster[avec_cluster["Modele"].isin(["Logistic Regression", "XGBoost"])].copy()
baseline_avec["Version"] = "Avant - Avec cluster"
# print(baseline_avec)


optimises = results_optimized.copy()
optimises["Version"] = "Après - " + optimises["Version"]

comparaison=pd.concat([baseline_avec,baseline_sans,optimises],ignore_index=True)

display(comparaison.round(4))

,Modele,Accuracy,Precision,Recall,F1-score,ROC-AUC,Version
0,Logistic Regression,0.7381,0.5042,0.7941,0.6168,0.8403,Avant - Avec cluster
1,XGBoost,0.7786,0.5807,0.5963,0.5884,0.8162,Avant - Avec cluster
2,Logistic Regression,0.7374,0.5034,0.7941,0.6162,0.8416,Avant -Sans cluster
3,SVM,0.7544,0.5273,0.7219,0.6095,0.8210,Avant -Sans cluster
4,Logistic Regression,0.7381,0.5042,0.7941,0.6168,0.8403,Après - Avec cluster
5,SVM,0.7445,0.5122,0.7861,0.6203,0.8289,Après - Avec cluster
6,Logistic Regression,0.7438,0.5113,0.7861,0.6196,0.8397,Après - Sans cluster
7,XGBoost,0.7708,0.5511,0.7353,0.6300,0.8428,Après - Sans cluster


In [40]:
best_model = grid_search.best_estimator_

print(grid_search.best_params_)

joblib.dump(
    best_model,
    "../models/best_logistic_regression_cluster.pkl"
)

{'model__C': 1}


['../models/best_logistic_regression_cluster.pkl']

In [ ]:
import sys
import importlib

sys.path.append("../src")

import mlflow_tracking
importlib.reload(mlflow_tracking)

from mlflow_tracking import enregistrer_experience
# parametres de preprocessing 
preprocessing = {
    "imputation": "median",
    "scaling": "StandardScaler",
    "encoding": "OneHotEncoder",
    "resampling": "SMOTE",
    "cluster_feature": "oui"
}

# experience 1 : Logistic Regression avec cluster
enregistrer_experience(
    model=grid_search.best_estimator_,
    X_test=X_test,
    y_test=y_test,
    params=grid_search.best_params_,
    nom_run="LogisticRegression_Cluster",
    preprocessing_params=preprocessing
)

# experience 2 : SVM avec cluster
enregistrer_experience(
    model=svm_grid_search.best_estimator_,
    X_test=X_test,
    y_test=y_test,
    params=svm_grid_search.best_params_,
    nom_run="SVM_Cluster",
    preprocessing_params=preprocessing
)


2026/10/10 11:58:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experience 'LogisticRegression_Cluster' enregistree.


2026/10/10 11:58:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experience 'SVM_Cluster' enregistree.
